In [41]:
import sys
import os

sys.path.append(os.path.abspath(".."))

In [42]:
import os
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score


JSON_DIR = "../json"
LABEL_DIR = "../json_label"

MODEL_PATH = "../src/ml_model/random_forest_special_assessment.pkl"


LEVEL_MAPPING = {
    "Không có": 0,
    "Chưa đủ cơ sở": 1,
    "Đang hình thành": 2,
    "Đạt": 3,
    "Nổi trội": 4
}

REVERSE_LEVEL_MAPPING = {
    0: "Không có",
    1: "Chưa đủ cơ sở",
    2: "Đang hình thành",
    3: "Đạt",
    4: "Nổi trội"
}

SUBJECTS = [
    "vietnamese",
    "mathematics",
    "informatics",
    "science",
    "history_and_geography",
    "english",
    "technology",
    "music",
    "arts",
    "civics",
    "physical_education",
    "experiential_activities"
    # "nature_and_society"
]

LABEL_FIELDS = [
    "Năng lực ngôn ngữ",
    "Năng lực tính toán",
    "Năng lực khoa học",
    "Năng lực công nghệ",
    "Năng lực tin học",
    "Năng lực thẩm mĩ",
    "Năng lực thể chất"
]

In [43]:
def load_input(json_path):
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    features = []
    for subject in SUBJECTS:
        subject_data = data.get(subject, {})
        confident = subject_data.get("confident", 0)
        level = subject_data.get("level", 0)
        features.append(confident)
        features.append(level)
    return features

In [44]:
load_input("../json/20252026.01.Mo1.001.json")

[0.8557479977607727,
 1,
 0.6805649995803833,
 1,
 0.6744087934494019,
 1,
 0,
 0,
 0,
 0,
 0.5206648707389832,
 1,
 0,
 0,
 0.6389163732528687,
 1,
 0.7888104915618896,
 1,
 0.6668856143951416,
 1,
 0.6903708577156067,
 1,
 0.7037678360939026,
 2]

In [45]:
def load_label(label_path):
    with open(label_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    label_dict = {
        item["field"]: LEVEL_MAPPING[item["level"]]
        for item in data["labels"]
    }
    labels = []
    for field in LABEL_FIELDS:
        labels.append(label_dict.get(field, 0))
    return labels

In [46]:
load_label("../json_label/20252026.01.Mo1.001_label.json")

[2, 2, 2, 1, 2, 2, 2]

In [47]:
def build_dataset():
    X = []
    y = []
    student_codes = []

    for filename in os.listdir(JSON_DIR):
        if not filename.endswith(".json"):
            continue
        student_code = filename.replace(".json", "")
        input_path = os.path.join(JSON_DIR, filename)
        label_path = os.path.join(LABEL_DIR, f"{student_code}_label.json")
        if not os.path.exists(label_path):
            print(f"Bỏ qua {student_code}: không tìm thấy label")
            continue
        features = load_input(input_path)
        labels = load_label(label_path)
        X.append(features)
        y.append(labels)
        student_codes.append(student_code)
    return np.array(X), np.array(y), student_codes
build_dataset()

(array([[0.855748  , 1.        , 0.680565  , ..., 1.        , 0.70376784,
         2.        ],
        [0.84934038, 1.        , 0.77300429, ..., 1.        , 0.70376784,
         2.        ],
        [0.855748  , 1.        , 0.77300429, ..., 3.        , 0.70376784,
         2.        ],
        ...,
        [0.86158067, 3.        , 0.64353263, ..., 3.        , 0.7526933 ,
         1.        ],
        [0.81169844, 2.        , 0.79678929, ..., 1.        , 0.7526933 ,
         1.        ],
        [0.84702122, 1.        , 0.78451395, ..., 3.        , 0.95487177,
         3.        ]], shape=(126, 24)),
 array([[2, 2, 2, 1, 2, 2, 2],
        [2, 2, 2, 1, 4, 2, 2],
        [2, 2, 2, 1, 2, 2, 4],
        [2, 2, 2, 1, 2, 2, 2],
        [2, 2, 2, 1, 2, 2, 2],
        [2, 2, 2, 1, 3, 2, 2],
        [2, 2, 2, 1, 2, 2, 4],
        [2, 2, 2, 1, 3, 2, 2],
        [2, 2, 2, 1, 2, 2, 2],
        [0, 0, 2, 1, 2, 2, 2],
        [0, 0, 2, 1, 3, 2, 2],
        [2, 2, 2, 1, 2, 2, 2],
        [2, 2, 2, 1,

In [48]:
def train_model():
    X, y, student_codes = build_dataset()
    print("Số lượng samples:", len(X))
    print("Số features:", X.shape[1])
    print("Số labels:", y.shape[1])
    if len(X) < 2:
        raise ValueError("Cần ít nhất 2 samples để train/test split.")
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42
    )
    base_model = RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        class_weight="balanced"
    )
    model = MultiOutputClassifier(base_model)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    print("\n=== Đánh giá từng năng lực ===")
    for i, field in enumerate(LABEL_FIELDS):
        print(f"\n--- {field} ---")
        print("Accuracy:", accuracy_score(y_test[:, i], y_pred[:, i]))
        print(
            classification_report(
                y_test[:, i],
                y_pred[:, i],
                labels=[0, 1, 2, 3, 4],
                target_names=[
                    "Không có",
                    "Chưa đủ cơ sở",
                    "Đang hình thành",
                    "Đạt",
                    "Nổi trội"
                ],
                zero_division=0
            )
        )

    joblib.dump(model, MODEL_PATH)
    print(f"\nĐã lưu model tại: {MODEL_PATH}")
    return model

In [49]:
def predict_one(model, json_path):
    features = load_input(json_path)
    prediction = model.predict([features])[0]
    result = []
    for field, level_id in zip(LABEL_FIELDS, prediction):
        result.append({
            "field": field,
            "level": REVERSE_LEVEL_MAPPING[int(level_id)]
        })
    return result

In [50]:
model = train_model()
test_file = "../json/20252026.01.Mo1.001.json"
if os.path.exists(test_file):
    result = predict_one(model, test_file)
    print("\n=== Kết quả dự đoán thử ===")
    print(json.dumps(result, ensure_ascii=False, indent=2))

Số lượng samples: 126
Số features: 24
Số labels: 7

=== Đánh giá từng năng lực ===

--- Năng lực ngôn ngữ ---
Accuracy: 0.7307692307692307
                 precision    recall  f1-score   support

       Không có       0.00      0.00      0.00         1
  Chưa đủ cơ sở       0.00      0.00      0.00         0
Đang hình thành       0.73      1.00      0.84        19
            Đạt       0.00      0.00      0.00         2
       Nổi trội       0.00      0.00      0.00         4

       accuracy                           0.73        26
      macro avg       0.15      0.20      0.17        26
   weighted avg       0.53      0.73      0.62        26


--- Năng lực tính toán ---
Accuracy: 0.8846153846153846
                 precision    recall  f1-score   support

       Không có       1.00      1.00      1.00         1
  Chưa đủ cơ sở       0.00      0.00      0.00         0
Đang hình thành       0.85      1.00      0.92        11
            Đạt       1.00      0.25      0.40         4
  

In [51]:
from joblib import load

test_file = "../json/20252026.01.Mo1.001.json"
if os.path.exists(test_file):
    model = load("../src/ml_model/random_forest_special_assessment.pkl")
    features = load_input(test_file)
    prediction = model.predict([features])[0]
    result = []
    for field, level_id in zip(LABEL_FIELDS, prediction):
        result.append({
            "field": field,
            "level": REVERSE_LEVEL_MAPPING[int(level_id)]
        })
result

[{'field': 'Năng lực ngôn ngữ', 'level': 'Đang hình thành'},
 {'field': 'Năng lực tính toán', 'level': 'Đang hình thành'},
 {'field': 'Năng lực khoa học', 'level': 'Đang hình thành'},
 {'field': 'Năng lực công nghệ', 'level': 'Chưa đủ cơ sở'},
 {'field': 'Năng lực tin học', 'level': 'Đang hình thành'},
 {'field': 'Năng lực thẩm mĩ', 'level': 'Đang hình thành'},
 {'field': 'Năng lực thể chất', 'level': 'Đang hình thành'}]

In [52]:
d = {}
d["key"] = "value"
d

{'key': 'value'}

In [53]:
d.update({
    "a": 1,
    "b": 2
})

In [54]:
d

{'key': 'value', 'a': 1, 'b': 2}

In [55]:
d.update({})
d

{'key': 'value', 'a': 1, 'b': 2}

In [56]:
a = {"including": 1, "level": 2}
a.update({"loc": 1})
a

{'including': 1, 'level': 2, 'loc': 1}

In [57]:
import sys
import os

sys.path.append(os.path.abspath(".."))
from src.model.mongo.comments import Comments


In [58]:
data = {'student_code': '20252026.01.Mo1.001', 'vietnamese_comment': {'evidence': 'biết đặt câu hoàn chỉnh', 'confident': 0.8557479977607727, 'indicator': 'Nói rõ ràng, thành câu.', 'level': 1, 'full_comment': 'Em đọc bài khá tốt, biết đặt câu hoàn chỉnh. Em cần rèn thêm chữ viết nhé!'}, 'mathematics_comment': {'evidence': 'vận dụng tốt vào các bài tập', 'confident': 0.6805649995803833, 'indicator': 'Củng cố và hoàn thiện các kĩ năng: đọc, viết, so sánh, xếp thứ tự được các số tự nhiên.', 'level': 1, 'full_comment': 'Em thực hiện tính đúng, vận dụng tốt vào các bài tập.'}, 'informatics_comment': {'evidence': 'em biết chọn các công cụ vẽ có sẵn của paint để vẽ các hình đơn giản', 'confident': 0.6744087934494019, 'indicator': 'Tạo được bài trình chiếu đơn giản, bưu thiệp, bức vẽ hay một chương trình trò chơi đơn giản.', 'level': 1, 'full_comment': 'Em biết chọn các công cụ vẽ có sẵn của Paint để vẽ các hình đơn giản.'}, 'science_comment': {'evidence': None, 'confident': 0, 'indicator': None, 'level': 0, 'full_comment': ''}, 'history_and_geography_comment': {'evidence': None, 'confident': 0, 'indicator': None, 'level': 0, 'full_comment': ''}, 'english_comment': {'evidence': 'em có tinh thần học tập tốt', 'confident': 0.5206648707389832, 'indicator': 'Có thái độ tích cực đối với việc học tiếng Anh.', 'level': 1, 'full_comment': 'Em có tinh thần học tập tốt. Em cần phát huy.'}, 'technology_comment': {'evidence': None, 'confident': 0, 'indicator': None, 'level': 0, 'full_comment': ''}, 'music_comment': {'evidence': 'theo phách', 'confident': 0.6389163732528687, 'indicator': 'Đọc đúng cao độ gam Đô trưởng.', 'level': 1, 'full_comment': 'Em biết gõ đệm theo tiết tấu lời ca, theo phách.'}, 'arts_comment': {'evidence': 'tô màu đều nét', 'confident': 0.7888104915618896, 'indicator': 'Sử dụng được các màu cơ bản; màu đậm, màu nhạt trong thực hành, sáng tạo.', 'level': 1, 'full_comment': 'Em biết trình bày các bố cục hài hòa và tô màu đều nét.'}, 'civics_comment': {'evidence': 'hành vi đạo đức phù hợp', 'confident': 0.6668856143951416, 'indicator': 'Nhận xét được tính chất đúng – sai, tốt – xấu, thiện – ác của một số thái độ, hành vi đạo đức và pháp luật của bản thân và bạn bè trong học tập và sinh hoạt.', 'level': 1, 'full_comment': 'Em biết điều chỉnh thái độ và hành vi đạo đức phù hợp.'}, 'physical_education_comment': {'evidence': 'em hoàn thành nội dung chương trình môn học', 'confident': 0.6903708577156067, 'indicator': 'Hoàn thành lượng vận động của bài tập.', 'level': 1, 'full_comment': 'Em hoàn thành nội dung chương trình môn học.'}, 'experiential_activities_comment': {'evidence': 'tham gia tốt các hoạt động', 'confident': 0.7037678360939026, 'indicator': 'Thực hiện và hoàn thành được các nhiệm vụ.', 'level': 2, 'full_comment': 'Em hiểu bài và tham gia tốt các hoạt động.'}}

In [59]:
feature = Comments(**data)

In [60]:
feature.to_dict()

{'student_code': '20252026.01.Mo1.001',
 'data': {'vietnamese_comment': {'evidence': 'biết đặt câu hoàn chỉnh',
   'confident': 0.8557479977607727,
   'indicator': 'Nói rõ ràng, thành câu.',
   'level': 1,
   'full_comment': 'Em đọc bài khá tốt, biết đặt câu hoàn chỉnh. Em cần rèn thêm chữ viết nhé!'},
  'mathematics_comment': {'evidence': 'vận dụng tốt vào các bài tập',
   'confident': 0.6805649995803833,
   'indicator': 'Củng cố và hoàn thiện các kĩ năng: đọc, viết, so sánh, xếp thứ tự được các số tự nhiên.',
   'level': 1,
   'full_comment': 'Em thực hiện tính đúng, vận dụng tốt vào các bài tập.'},
  'informatics_comment': {'evidence': 'em biết chọn các công cụ vẽ có sẵn của paint để vẽ các hình đơn giản',
   'confident': 0.6744087934494019,
   'indicator': 'Tạo được bài trình chiếu đơn giản, bưu thiệp, bức vẽ hay một chương trình trò chơi đơn giản.',
   'level': 1,
   'full_comment': 'Em biết chọn các công cụ vẽ có sẵn của Paint để vẽ các hình đơn giản.'},
  'science_comment': {'ev